# TrustVis — camera-ready analysis from the **submitted run's saved outputs** (no retraining, no GPU)

Uses the per-image pickles written by the submitted notebook
(`trustvis_results/attributions/{ff_test,celeb_df}_{model}_gradcam_results.pkl`). Each entry
already stores the prediction, label, the Grad-CAM++ map and the face-parsing map, so we can:

1. **confirm** why Table 1 rows for ViT-B/16 and MobileNetV2 sum to 45.3 % / 54.8 % (count empty maps),
2. recompute Table 1 with empty maps excluded, and add the **area-normalised** attribution (reviewer rYMW),
3. recompute SAC under **both** definitions (per-image mean — what Table 3 contains — and Eq. 3 as written),
4. test whether high SAC implies **pixel-level** agreement or **shared errors** (reviewer rYMW, meta-review),
5. redo the non-facial vs. accuracy-drop correlation with p-value, bootstrap CI and leave-one-out,
6. report the attribution protocol subsets (all images / correctly classified FAKE only).

If the Kaggle session that produced the pickles is gone: open that notebook version → *Output* →
*New dataset* (or *Add input* from this notebook) and point `ATTR_DIR` at it.

In [ ]:
import os, glob, gc
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

ATTR_DIR = "/kaggle/working/trustvis_results/attributions"     # <- edit if attached as an input dataset
OUT_DIR = "/kaggle/working/trustvis_camera_ready_from_submitted_run"
os.makedirs(f"{OUT_DIR}/tables", exist_ok=True)
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
MODEL_NAMES = ['xception', 'efficientnet_b4', 'vit_base_patch16_224', 'resnet50',
               'vgg16', 'vgg19', 'mobilenetv2_100', 'inception_resnet_v2']
SAL_STORE = 56
EMPTY_EPS = 1e-8
print(sorted(os.path.basename(p) for p in glob.glob(f"{ATTR_DIR}/*_results.pkl")))

In [ ]:
# ==================== ANALYSIS CORE (shared by both notebooks) ====================
# Every analysis works on one "record" per (dataset, model), a dict of aligned numpy arrays:
#   paths (N,)  label (N,)  pred (N,)  prob_fake (N,)
#   R      (N,6)  relative attribution in %, NaN rows where the saliency map was empty
#   valid  (N,)   False where Grad-CAM++/IG returned an all-zero map
#   sal    (N,s,s) float16 saliency map stored at low resolution (for spatial agreement)
# and one parse dict per dataset: region_idx (N,H,W) uint8 in 0..5, area (N,6) in %.
# Images are in the same order for every model of a dataset.
import numpy as np
import pandas as pd
from scipy import stats

REGION_NAMES = ['eyes', 'nose', 'mouth', 'skin', 'hair', 'background']
FACIAL_FEATURES = ['eyes', 'nose', 'mouth']
NON_FACIAL = ['hair', 'background']
PRETTY = {
    'xception': 'Xception', 'efficientnet_b4': 'EfficientNet-B4', 'vit_base_patch16_224': 'ViT-B/16',
    'resnet50': 'ResNet50', 'vgg16': 'VGG16', 'vgg19': 'VGG19', 'mobilenetv2_100': 'MobileNetV2',
    'inception_resnet_v2': 'Inception-ResNet-v2',
}
_RI = {r: i for i, r in enumerate(REGION_NAMES)}


def _idx(names):
    return [_RI[n] for n in names]


def _cos_rows(a, b):
    na, nb = np.linalg.norm(a, axis=1), np.linalg.norm(b, axis=1)
    return (a * b).sum(1) / (na * nb)


def _cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def _auc(label, score):
    try:
        from sklearn.metrics import roc_auc_score
        if len(np.unique(label)) < 2:
            return np.nan
        return 100.0 * roc_auc_score(label, score)
    except Exception:
        return np.nan


def subset_mask(rec, which):
    """which: 'all' | 'fake' | 'real' | 'correct' | 'correct_fake' | 'wrong'."""
    lab, pred = rec['label'], rec['pred']
    return {
        'all': np.ones(len(lab), bool),
        'fake': lab == 1,
        'real': lab == 0,
        'correct': pred == lab,
        'correct_fake': (pred == lab) & (lab == 1),
        'wrong': pred != lab,
    }[which]


def summarize_regions(records, parse, which='all'):
    """Table 1 replacement.  R_k: mean relative attribution over images with a non-empty map
    (each row sums to 100).  A_k: mean area share of the region.  E_k = R_k / A_k is the
    area-normalised attribution (1.0 = the region gets exactly its pixel share; >1 = it gets
    more saliency per pixel than average).  E_k is proportional to the paper's Eq. (1) A_{d,k}."""
    rows = []
    for m, r in records.items():
        sel = subset_mask(r, which)
        v = r['valid'] & sel
        n_sel = int(sel.sum())
        row = {'model': m, 'subset': which, 'n_images': n_sel, 'n_valid_maps': int(v.sum()),
               'empty_map_rate_%': 100.0 * (1 - v.sum() / max(n_sel, 1)),
               'accuracy_%': 100.0 * float((r['pred'][sel] == r['label'][sel]).mean()) if n_sel else np.nan,
               'auc_%': _auc(r['label'][sel], r['prob_fake'][sel]) if n_sel else np.nan}
        if v.sum() == 0:
            rows.append(row)
            continue
        Rm = np.nanmean(r['R'][v], axis=0)
        # what the submitted Table 1 reported: empty maps counted as 0 % everywhere
        legacy = np.where(r['valid'][:, None], np.nan_to_num(r['R']), 0.0)[sel].mean(0)
        Am = parse['area'][v].mean(0)
        E = Rm / np.maximum(Am, 1e-9)
        for k, name in enumerate(REGION_NAMES):
            row[f'R_{name}'] = Rm[k]
            row[f'A_{name}'] = Am[k]
            row[f'E_{name}'] = E[k]
            row[f'legacyR_{name}'] = legacy[k]
        row['R_sum'] = Rm.sum()
        row['legacyR_sum'] = legacy.sum()
        row['R_non_facial'] = Rm[_idx(NON_FACIAL)].sum()
        row['R_features'] = Rm[_idx(FACIAL_FEATURES)].sum()
        row['A_non_facial'] = Am[_idx(NON_FACIAL)].sum()
        row['E_non_facial'] = row['R_non_facial'] / max(row['A_non_facial'], 1e-9)
        row['A_features'] = Am[_idx(FACIAL_FEATURES)].sum()
        row['E_features'] = row['R_features'] / max(row['A_features'], 1e-9)
        row['legacyR_non_facial'] = legacy[_idx(NON_FACIAL)].sum()
        rows.append(row)
    return pd.DataFrame(rows)


def pairwise_tables(records, top_frac=0.2):
    """SAC under both definitions + spatial agreement + error overlap, for all model pairs.
    SAC_per_image : mean over images of cos(R_d1(x), R_d2(x))   <- what the submitted Table 3 contains
    SAC_global    : cos(mean_x R_d1, mean_x R_d2)                <- what the submitted Eq. (3) states
    spatial_r     : mean per-image Pearson r between the two saliency maps (pixel level)
    top_iou       : mean per-image IoU of the top-`top_frac` most salient pixels
    err_jaccard   : |E1 & E2| / |E1 | E2| over misclassified images
    kappa_correct : Cohen's kappa between the two models' correct/incorrect vectors."""
    models = list(records)
    out = []
    for i, m1 in enumerate(models):
        for m2 in models[i + 1:]:
            r1, r2 = records[m1], records[m2]
            assert np.array_equal(r1['paths'], r2['paths']), 'records must be image-aligned'
            both = r1['valid'] & r2['valid']
            cs = _cos_rows(r1['R'][both], r2['R'][both])
            g1 = np.nanmean(r1['R'][r1['valid']], 0)
            g2 = np.nanmean(r2['R'][r2['valid']], 0)
            s1 = r1['sal'][both].reshape(both.sum(), -1).astype(np.float32)
            s2 = r2['sal'][both].reshape(both.sum(), -1).astype(np.float32)
            pr, ious = [], []
            k = max(1, int(round(top_frac * s1.shape[1]))) if len(s1) else 1
            for a, b in zip(s1, s2):
                if a.std() > 0 and b.std() > 0:
                    pr.append(np.corrcoef(a, b)[0, 1])
                ta = np.zeros(a.shape, bool); ta[np.argpartition(-a, k - 1)[:k]] = True
                tb = np.zeros(b.shape, bool); tb[np.argpartition(-b, k - 1)[:k]] = True
                ious.append((ta & tb).sum() / (ta | tb).sum())
            c1 = r1['pred'] == r1['label']
            c2 = r2['pred'] == r2['label']
            e1, e2 = ~c1, ~c2
            union = (e1 | e2).sum()
            try:
                from sklearn.metrics import cohen_kappa_score
                kappa = cohen_kappa_score(c1, c2) if (len(np.unique(c1)) > 1 or len(np.unique(c2)) > 1) else np.nan
            except Exception:
                kappa = np.nan
            out.append({
                'model_1': m1, 'model_2': m2, 'n_both_valid': int(both.sum()),
                'SAC_per_image': float(cs.mean()) if len(cs) else np.nan,
                'SAC_per_image_std': float(cs.std()) if len(cs) else np.nan,
                'SAC_global': _cos(g1, g2),
                'top_region_agree_%': 100.0 * float((r1['R'][both].argmax(1) == r2['R'][both].argmax(1)).mean()) if both.any() else np.nan,
                'spatial_r': float(np.mean(pr)) if pr else np.nan,
                f'top{int(top_frac * 100)}_iou': float(np.mean(ious)) if ious else np.nan,
                'err_jaccard': float((e1 & e2).sum() / union) if union else np.nan,
                'both_wrong_%': 100.0 * float((e1 & e2).mean()),
                'kappa_correct': kappa,
            })
    return pd.DataFrame(out)


def pair_matrix(df_pairs, col, models):
    M = np.ones((len(models), len(models)))
    for _, row in df_pairs.iterrows():
        i, j = models.index(row['model_1']), models.index(row['model_2'])
        M[i, j] = M[j, i] = row[col]
    return pd.DataFrame(M, index=[PRETTY.get(m, m) for m in models], columns=[PRETTY.get(m, m) for m in models])


def corr_report(x, y, n_boot=10000, seed=0, label=''):
    """Pearson + Spearman + bootstrap 95% CI + leave-one-detector-out range (n is tiny: be honest)."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    pr, pp = stats.pearsonr(x, y)
    sr, sp = stats.spearmanr(x, y)
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        i = rng.integers(0, len(x), len(x))
        if len(np.unique(i)) < 3 or np.std(x[i]) == 0 or np.std(y[i]) == 0:
            continue
        boots.append(np.corrcoef(x[i], y[i])[0, 1])
    loo = [np.corrcoef(np.delete(x, j), np.delete(y, j))[0, 1] for j in range(len(x))]
    return {'what': label, 'n': len(x), 'pearson_r': pr, 'pearson_p': pp, 'spearman_rho': sr, 'spearman_p': sp,
            'boot95_lo': np.percentile(boots, 2.5), 'boot95_hi': np.percentile(boots, 97.5),
            'loo_min': min(loo), 'loo_max': max(loo)}


def generalisation_table(df_src, df_tgt):
    a = df_src[['model', 'accuracy_%', 'auc_%', 'R_non_facial', 'E_non_facial', 'legacyR_non_facial', 'R_skin', 'E_skin', 'empty_map_rate_%']]
    b = df_tgt[['model', 'accuracy_%', 'auc_%']]
    d = a.merge(b, on='model', suffixes=('_src', '_tgt'))
    d['acc_drop_pp'] = d['accuracy_%_src'] - d['accuracy_%_tgt']
    d['auc_drop_pp'] = d['auc_%_src'] - d['auc_%_tgt']
    return d


def correlation_suite(gen):
    rows = []
    for xcol, lab in [('legacyR_non_facial', 'non-facial R as submitted (empty maps = 0)'),
                      ('R_non_facial', 'non-facial R, empty maps excluded'),
                      ('E_non_facial', 'non-facial area-normalised E'),
                      ('R_skin', 'skin R, empty maps excluded')]:
        for ycol in ['acc_drop_pp', 'auc_drop_pp']:
            if gen[ycol].notna().all():
                rows.append(corr_report(gen[xcol], gen[ycol], label=f'{lab} vs {ycol}'))
    return pd.DataFrame(rows)


def latex_table1(df, caption_note=''):
    lines = [r'\begin{tabular}{l' + 'c' * 9 + '}', r'\toprule',
             r'Detector & Eyes & Nose & Mouth & Skin & Hair & Bg & Non-facial & Features & Empty (\%) \\',
             r'\midrule']
    for _, r in df.iterrows():
        cells = [f"{r[f'R_{k}']:.1f} ({r[f'E_{k}']:.2f})" for k in REGION_NAMES]
        cells += [f"{r['R_non_facial']:.1f} ({r['E_non_facial']:.2f})", f"{r['R_features']:.1f} ({r['E_features']:.2f})",
                  f"{r['empty_map_rate_%']:.1f}"]
        lines.append(PRETTY.get(r['model'], r['model']) + ' & ' + ' & '.join(cells) + r' \\')
    lines += [r'\bottomrule', r'\end{tabular}']
    return '\n'.join(lines)


def latex_pairs(df_pairs, cols=('SAC_per_image', 'SAC_global', 'spatial_r', 'top20_iou', 'err_jaccard')):
    head = 'Detector pair & ' + ' & '.join(c.replace('_', r'\_') for c in cols) + r' \\'
    lines = [r'\begin{tabular}{l' + 'c' * len(cols) + '}', r'\toprule', head, r'\midrule']
    for _, r in df_pairs.iterrows():
        lines.append(f"{PRETTY.get(r['model_1'], r['model_1'])} vs. {PRETTY.get(r['model_2'], r['model_2'])} & "
                     + ' & '.join(f'{r[c]:.3f}' for c in cols) + r' \\')
    lines += [r'\bottomrule', r'\end{tabular}']
    return '\n'.join(lines)

In [ ]:
# CelebAMask-HQ ids -> the paper's 6 regions (identical to the submitted REGION_MAP)
REGION_MAP = {'eyes': [2, 3, 4, 5, 6], 'nose': [10], 'mouth': [11, 12, 13], 'skin': [1], 'hair': [17],
              'background': [0, 7, 8, 9, 14, 15, 16, 18]}
CLASS_TO_REGION = np.zeros(19, dtype=np.uint8)
for k, name in enumerate(REGION_NAMES):
    CLASS_TO_REGION[REGION_MAP[name]] = k


def load_submitted(tag):
    """Convert the submitted pickles into image-aligned records (one model in memory at a time)."""
    per_model, parse_ref, order = {}, None, None
    for m in MODEL_NAMES:
        f = f"{ATTR_DIR}/{tag}_{m}_gradcam_results.pkl"
        if not os.path.exists(f):
            print(f"  missing {f}")
            continue
        lst = pd.read_pickle(f)
        if order is None:
            order = [r['img_path'] for r in lst]
            parse_ref = {r['img_path']: r['parsing_mask'] for r in lst}
        by_path = {r['img_path']: r for r in lst}
        N = len(order)
        rec = {'paths': np.array(order), 'label': np.zeros(N, int), 'pred': np.zeros(N, int),
               'prob_fake': np.full(N, np.nan, np.float32), 'R': np.full((N, 6), np.nan),
               'valid': np.zeros(N, bool), 'sal': np.zeros((N, SAL_STORE, SAL_STORE), np.float16),
               'legacy_R_sum': np.zeros(N), 'present': np.zeros(N, bool)}
        for i, p in enumerate(order):
            r = by_path.get(p)
            if r is None:
                continue
            rec['present'][i] = True
            rec['label'][i], rec['pred'][i] = r['true_label'], r['prediction']
            # submitted pickles store the confidence of the predicted class
            rec['prob_fake'][i] = r['confidence'] if r['prediction'] == 1 else 1 - r['confidence']
            s = np.asarray(r['saliency_map'], np.float64)
            legacy = np.array([r['region_attributions'][k] for k in REGION_NAMES])
            rec['legacy_R_sum'][i] = legacy.sum()
            if np.isfinite(s.sum()) and s.sum() > EMPTY_EPS:
                rec['valid'][i] = True
                rec['R'][i] = legacy / legacy.sum() * 100.0          # identical to recomputing from the map
            rec['sal'][i] = cv2.resize(s.astype(np.float32), (SAL_STORE, SAL_STORE), interpolation=cv2.INTER_AREA)
        per_model[m] = rec
        del lst, by_path
        gc.collect()
        print(f"  {tag} {m}: {rec['present'].sum()} images, empty maps {100 * (1 - rec['valid'][rec['present']].mean()):.1f}%")
    keep = np.all([r['present'] for r in per_model.values()], axis=0)
    for m in per_model:
        per_model[m] = {k: v[keep] for k, v in per_model[m].items()}
    region_idx = np.stack([CLASS_TO_REGION[parse_ref[p]] for p in np.array(order)[keep]])
    area = np.stack([np.bincount(x.ravel(), minlength=6) / x.size * 100 for x in region_idx])
    return per_model, {'region_idx': region_idx, 'area': area}


records, parses = {}, {}
for tag in ['ff_test', 'celeb_df']:
    records[tag], parses[tag] = load_submitted(tag)

## 1. Diagnosis: why do the ViT-B/16 and MobileNetV2 rows not sum to 100 %?

In [ ]:
diag = []
for m, r in records['ff_test'].items():
    s = r['legacy_R_sum']
    diag.append({'model': m, 'n': len(s), 'empty_maps': int((~r['valid']).sum()),
                 'empty_%': 100 * (~r['valid']).mean(),
                 'images_with_sum≈100': int((np.abs(s - 100) < 1e-3).sum()),
                 'images_with_sum=0': int((s == 0).sum()),
                 'other_sums': int(((np.abs(s - 100) >= 1e-3) & (s != 0)).sum()),
                 'Table1_row_sum_as_submitted': s.mean()})
diag = pd.DataFrame(diag)
diag.to_csv(f"{OUT_DIR}/tables/diagnosis_empty_maps.csv", index=False)
print(diag.round(2).to_string(index=False))
print("\nExpected: every image sums to either 100 % (valid map) or 0 % (empty map), and "
      "Table1_row_sum = 100 × (1 − empty rate). If 'other_sums' is not 0, tell me — the diagnosis would need revisiting.")

## 2. Table 1 corrected (+ area-normalised attribution)

In [ ]:
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 80)
df_ff = summarize_regions(records['ff_test'], parses['ff_test'], 'all')
df_cd = summarize_regions(records['celeb_df'], parses['celeb_df'], 'all')
df_ff.to_csv(f"{OUT_DIR}/tables/table1_ff_all.csv", index=False)
print("As submitted (empty maps counted as 0 %) — must reproduce the paper's Table 1:")
print(df_ff[['model'] + [f'legacyR_{k}' for k in REGION_NAMES] + ['legacyR_sum', 'legacyR_non_facial']].round(1).to_string(index=False))
print("\nCorrected R (%), empty maps excluded:")
print(df_ff[['model', 'accuracy_%', 'empty_map_rate_%'] + [f'R_{k}' for k in REGION_NAMES] + ['R_sum', 'R_non_facial', 'R_features']].round(1).to_string(index=False))
print("\nArea share A (%) and area-normalised E = R/A:")
print(df_ff[['model'] + [f'A_{k}' for k in REGION_NAMES] + [f'E_{k}' for k in REGION_NAMES] + ['E_non_facial', 'E_features']].round(2).to_string(index=False))
with open(f"{OUT_DIR}/tables/table1_ff_all.tex", 'w') as f:
    f.write(latex_table1(df_ff))

for which in ['correct_fake', 'fake', 'real']:
    d = summarize_regions(records['ff_test'], parses['ff_test'], which)
    d.to_csv(f"{OUT_DIR}/tables/table1_ff_{which}.csv", index=False)
    print(f"\nSubset = {which}:")
    print(d[['model', 'n_images', 'empty_map_rate_%'] + [f'R_{k}' for k in REGION_NAMES] + ['R_non_facial', 'E_non_facial']].round(2).to_string(index=False))

## 3. Generalisation vs. non-facial attribution, with honest statistics

In [ ]:
gen = generalisation_table(df_ff, df_cd)
gen.to_csv(f"{OUT_DIR}/tables/table2_generalisation.csv", index=False)
print(gen[['model', 'accuracy_%_src', 'accuracy_%_tgt', 'acc_drop_pp', 'auc_%_src', 'auc_%_tgt', 'auc_drop_pp',
           'legacyR_non_facial', 'R_non_facial', 'E_non_facial']].round(2).to_string(index=False))
corr = correlation_suite(gen)
corr.to_csv(f"{OUT_DIR}/tables/correlations.csv", index=False)
print(corr.round(3).to_string(index=False))

## 4. SAC (both definitions), pixel-level agreement, shared errors

In [ ]:
pairs_ff = pairwise_tables(records['ff_test'])
pairs_cd = pairwise_tables(records['celeb_df'])
pairs_ff.to_csv(f"{OUT_DIR}/tables/table3_pairs_ff.csv", index=False)
pairs_cd.to_csv(f"{OUT_DIR}/tables/pairs_celebdf.csv", index=False)
print(pairs_ff.round(3).to_string(index=False))
print("\nCeleb-DF error overlap:")
print(pairs_cd[['model_1', 'model_2', 'err_jaccard', 'both_wrong_%', 'kappa_correct']].round(3).to_string(index=False))
print(pd.DataFrame([corr_report(pairs_ff['SAC_per_image'], pairs_ff['spatial_r'], label='SAC vs pixel-level r (FF++)'),
                    corr_report(pairs_ff['SAC_per_image'], pairs_cd['err_jaccard'], label='SAC(FF++) vs error Jaccard (Celeb-DF)')]).round(3).to_string(index=False))
with open(f"{OUT_DIR}/tables/table3_pairs_ff.tex", 'w') as f:
    f.write(latex_pairs(pairs_ff))

In [ ]:
sns.set_style("whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
for a, col, ttl in zip(axes, ['SAC_per_image', 'SAC_global', 'spatial_r'],
                       ['SAC (mean per-image cosine)', 'SAC of mean vectors (Eq. 3 as submitted)', 'Pixel-level Pearson r']):
    sns.heatmap(pair_matrix(pairs_ff, col, MODEL_NAMES), annot=True, fmt='.2f', cmap='YlOrRd', vmin=0, vmax=1, square=True, ax=a, cbar=False)
    a.set_title(ttl)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/figures/sac_heatmaps.png", dpi=300, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
x, y = gen['R_non_facial'].values, gen['acc_drop_pp'].values
ax.scatter(x, y, s=120, edgecolors='k', zorder=3)
for xi, yi, m in zip(x, y, gen['model']):
    ax.annotate(PRETTY.get(m, m), (xi, yi), textcoords='offset points', xytext=(6, 4), fontsize=10)
c = corr_report(x, y)
xl = np.linspace(x.min() - 2, x.max() + 2, 50)
ax.plot(xl, np.polyval(np.polyfit(x, y, 1), xl), 'k--', alpha=0.6, label=f"r = {c['pearson_r']:.2f} (p = {c['pearson_p']:.3f}, n = {len(x)})")
ax.set_xlabel('Non-facial attribution on FF++ (hair + background, %), empty maps excluded')
ax.set_ylabel('Accuracy drop FF++ → Celeb-DF (pp)')
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/figures/cross_dataset_drop_corrected.png", dpi=300, bbox_inches='tight')
plt.show()
print("All outputs in", OUT_DIR)